In [3]:
import torch
import torch.nn as nn
from transformer_modules import *

class EncoderBlock(nn.Module):
    """Transformer 编码器单层块"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, ffn_num_input, ffn_num_hiddens, num_heads,
                 dropout, bias=False):
        super().__init__()
        self.attention = MultiHeadAttention(key_size, query_size, value_size, 
                                            num_hiddens, num_heads, dropout, bias)
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout)

    def forward(self, X, valid_lens):
        # 1. 经过多头自注意力 + AddNorm
        Y = self.addnorm1(X, self.attention(X, X, X, valid_lens))
        # 2. 经过前馈网络 + AddNorm
        return self.addnorm2(Y, self.ffn(Y))


# ==================== 单元测试验证 ====================
X = torch.ones((2, 100, 24)) # 模拟 2 句话，每句 100 个词，每个词 24 维
valid_lens = torch.tensor([3, 2])

encoder_blk = EncoderBlock(key_size=24, query_size=24, value_size=24, num_hiddens=24,
                           norm_shape=[100, 24], ffn_num_input=24, ffn_num_hiddens=48,
                           num_heads=8, dropout=0.5)
encoder_blk.eval()
out = encoder_blk(X, valid_lens)

print("=== EncoderBlock 测试成功 ===")
print("输入形状:", X.shape)   # [2, 100, 24]
print("输出形状:", out.shape) # [2, 100, 24] (完美保持输入形状不变！)


=== EncoderBlock 测试成功 ===
输入形状: torch.Size([2, 100, 24])
输出形状: torch.Size([2, 100, 24])


In [4]:
class TransformerEncoder(nn.Module):
    """Transformer 完整编码器大楼"""
    def __init__(self, vocab_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, num_layers, dropout, bias=False):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(vocab_size, num_hiddens)
        self.pos_encoding = PositionalEncoding(num_hiddens, dropout)
        
        # 堆叠 num_layers 个 EncoderBlock (比如 6 层)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block" + str(i),
                EncoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape, ffn_num_input, ffn_num_hiddens,
                             num_heads, dropout, bias))

    def forward(self, X, valid_lens):
        # 1. 词嵌入并乘上 sqrt(d) 放大方差，加上位置编码
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.num_hiddens))
        
        # 2. 依次穿过 6 层 EncoderBlock
        for blk in self.blks:
            X = blk(X, valid_lens)
        return X